# EVADE Pilot Benchmark Matrix
### Evaluation Awareness & Behavioral Shift Evaluation

This notebook runs the 200-task $\times$ 6-condition pilot benchmark matrix (1,200 evaluation instances).
It runs **completely offline on local GPU** using open-weights models (e.g. `Qwen/Qwen2.5-7B-Instruct` via 4-bit NF4 quantization) with **zero API keys, zero rate limits, and 100% reproducible execution**.

### Pipeline Overview
1. **Environment Setup**: Install `transformers`, `accelerate`, `bitsandbytes`, `sqlite-utils`, `matplotlib`, `seaborn`.
2. **Repository Sync**: Clone or update `researchpaper2-` with robust absolute pathing.
3. **Offline GPU Verification**: Check GPU accelerator and VRAM (no API keys required).
4. **Offline Benchmark Execution**: Run 200 tasks across 6 conditions (C0 to C5).
5. **Statistical Diagnostics**: Compute EBS, verbosity inflation, and test for prompt-length confounding.
6. **Figure Generation**: Automatically render high-res publication figures.
7. **Guarded Packaging**: Package raw JSONL, per-task pairs, summary, DB, and figures to `evade_pilot_results.zip`.

### Step 1: Install Dependencies & Verify GPU

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch torchvision scipy tabulate sqlite-utils matplotlib seaborn pandas

import torch
import importlib.metadata

print("PyTorch version:   ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:        ", torch.cuda.get_device_name(0))
    print("Device Count:      ", torch.cuda.device_count())
    print("Total VRAM (GB):   ", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
print("sqlite-utils:      ", importlib.metadata.version('sqlite-utils'))
print("[OK] Dependencies installed successfully!")

### Step 2: Sync EVADE Research Repository

In [ ]:
import os
import sys

REPO_DIR = "/kaggle/working/researchpaper2-"
EVADE_DIR = "/kaggle/working/researchpaper2-/evade"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Goldypahal/researchpaper2-.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin main

%cd {EVADE_DIR}
if EVADE_DIR not in sys.path:
    sys.path.insert(0, EVADE_DIR)
print("Active working directory:", os.getcwd())

### Step 3: Verify GPU Hardware (100% Offline Execution)

In [ ]:
import torch

print("=" * 65)
print("  OFFLINE LOCAL GPU VERIFICATION")
print("=" * 65)
assert torch.cuda.is_available(), "GPU accelerator required! In Kaggle: Notebook Settings -> Accelerator -> GPU T4 x2 or P100."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
print(f"Active GPU:     {gpu_name}")
print(f"Total VRAM:     {vram_gb} GB")
print("Execution Mode: 100% OFFLINE Open-Weights HuggingFace Execution")
print("Network Status: NO external API keys or API endpoints used.")
print("=" * 65)

### Step 4: Execute EVADE Pilot Matrix Completely Offline on GPU

Runs `Qwen/Qwen2.5-7B-Instruct` with 4-bit NF4 quantization on Kaggle GPU.
- 200 tasks $\times$ 6 conditions = 1,200 generations
- Zero rate limits, zero API dependencies, fast local inference.

In [ ]:
# Run full 200-task matrix offline with 4-bit quantization
!python scripts/run_pilot_matrix.py --model Qwen/Qwen2.5-7B-Instruct --quantize-4bit

### Step 5: Empirical Diagnostic & Confounding Inspection
Computes per-condition EBS, verbosity inflation, prompt-length correlation check ($r$), and task consistency across domains.

In [ ]:
!python scripts/inspect_results.py --dir pilot_results --model Qwen/Qwen2.5-7B-Instruct

### Step 6: Generate Publication Figures & Visualizations

In [ ]:
!python scripts/generate_pilot_figures.py --dir pilot_results --model Qwen/Qwen2.5-7B-Instruct

### Step 7: Guarded Packaging of Complete Research Artifacts

In [ ]:
import shutil
from pathlib import Path

raw_dir = Path("pilot_results/raw")
raw_files = list(raw_dir.glob("*.jsonl")) if raw_dir.exists() else []
total_records = sum(sum(1 for line in open(rf, encoding="utf-8") if line.strip()) for rf in raw_files)

print(f"Total raw records to package: {total_records}")
if total_records == 0:
    raise RuntimeError("CRITICAL: Experiment produced 0 generations. Refusing to package empty results!")

# Copy SQLite DB into pilot_results for packaging
db_file = Path("results/evade_results.db")
if db_file.exists():
    shutil.copy2(db_file, "pilot_results/")

out_zip = "/kaggle/working/evade_pilot_results"
shutil.make_archive(out_zip, "zip", "pilot_results")
print(f"[SUCCESS] All artifacts (raw JSONL, per-task pairs, summary, DB, figures) packaged to {out_zip}.zip!")